In [1]:
!pip install langchain langchain-community langchain-openai faiss-cpu tiktoken youtube-transcript-api deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1

In [3]:
import os
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

In [5]:
!pip install --upgrade youtube-transcript-api

In [6]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from deep_translator import GoogleTranslator

def get_transcript(video_id):
    try:
        # Try English first
        ytt = YouTubeTranscriptApi()
        transcript_list = ytt.fetch(video_id)
        transcript = " ".join(chunk.text for chunk in transcript_list)
        print("English transcript found!")
        return transcript

    except TranscriptsDisabled:
        print("Transcripts are disabled for this video.")
        return None

    except Exception as e:
        print(f"English not found. Trying other languages... {e}")
        try:
            ytt = YouTubeTranscriptApi()
            available = ytt.list(video_id)

            for t in available:
                print(f"Found: {t.language} ({t.language_code})")
                raw = t.fetch()
                raw_text = " ".join(chunk.text for chunk in raw)

                # Translate in chunks (4500 char limit)
                translator = GoogleTranslator(source='auto', target='en')
                parts = [raw_text[i:i+4500] for i in range(0, len(raw_text), 4500)]
                transcript = " ".join(translator.translate(p) for p in parts)
                print("Translated to English successfully!")
                return transcript

        except Exception as e2:
            print(f"Error: {e2}")
            return None

# Test it
video_id = "Gfr50f6ZBvo"
transcript = get_transcript(video_id)

if transcript:
    print(transcript[:500])
else:
    print("Could not get transcript.")

English transcript found!
the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful 


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_transcript(transcript):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
        separators=["\n\n", "\n", ".", " "]
    )
    chunks = splitter.create_documents([transcript])
    print(f"Total chunks: {len(chunks)}")
    return chunks

chunks = split_transcript(transcript)
print(chunks[0])

Total chunks: 207
page_content='the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please'


In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

def create_vector_store(chunks):
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vector_store = FAISS.from_documents(chunks, embeddings)
    print("Vector store created!")
    return vector_store

vector_store = create_vector_store(chunks)

Vector store created!


In [9]:
def create_retriever(vector_store):
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5}
    )
    print("Retriever created!")
    return retriever

retriever = create_retriever(vector_store)

Retriever created!


In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def create_rag_chain(retriever):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

    prompt = PromptTemplate(
        template="""
        You are an expert assistant analyzing a YouTube video transcript.
        Answer the question based ONLY on the provided transcript context.
        Be concise, clear and structured in your response.
        If the answer is not found, say "This topic was not covered in the video."

        Context:
        {context}

        Question: {question}

        Answer:
        """,
        input_variables=['context', 'question']
    )

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    chain = (
        RunnableParallel({
            'context': retriever | RunnableLambda(format_docs),
            'question': RunnablePassthrough()
        })
        | prompt | llm | StrOutputParser()
    )

    print("RAG chain created!")
    return chain

chain = create_rag_chain(retriever)

RAG chain created!


In [11]:
def ask(question):
    print(f"\nQuestion: {question}")
    answer = chain.invoke(question)
    print(f"Answer: {answer}")
    return answer

# Test questions
ask("What is the main topic of this video?")
ask("Who is Demis Hassabis?")
ask("What is DeepMind?")


Question: What is the main topic of this video?
Answer: The main topic of the video revolves around the exploration of artificial intelligence (AI), particularly the balance between science and engineering in solving intelligence, the pursuit of knowledge, and the potential of AI in addressing significant challenges like energy and climate. It also touches on philosophical questions about human existence and the nature of sentience in relation to AI.

Question: Who is Demis Hassabis?
Answer: Demis Hassabis is the CEO and co-founder of DeepMind, a company known for developing advanced artificial intelligence systems, including AlphaZero, which learned to play the game of Go better than any human, and AlphaFold 2, which solved the problem of protein folding. He is widely regarded as one of the most brilliant and impactful figures in the fields of artificial intelligence, science, and engineering.

Question: What is DeepMind?
Answer: DeepMind is a company that builds and publishes advanc

'DeepMind is a company that builds and publishes advanced artificial intelligence systems. It is known for creating significant AI technologies, such as AlphaGo, which learned to play the game of Go better than any human, and AlphaFold 2, which solved the complex problem of protein folding. DeepMind combines cutting-edge knowledge from various fields, including neuroscience, machine learning, engineering, mathematics, and philosophy, to foster innovation and invention in AI.'

In [12]:
def build_rag_pipeline(video_id):
    print(f"Building RAG pipeline for video: {video_id}")

    # Step 1 - Get transcript
    transcript = get_transcript(video_id)
    if not transcript:
        return None

    # Step 2 - Split
    chunks = split_transcript(transcript)

    # Step 3 - Vector store
    vector_store = create_vector_store(chunks)

    # Step 4 - Retriever
    retriever = create_retriever(vector_store)

    # Step 5 - Chain
    chain = create_rag_chain(retriever)

    print("\n✅ RAG Pipeline ready! You can now ask questions.")
    return chain

# Build pipeline
chain = build_rag_pipeline("Gfr50f6ZBvo")

# Ask anything
ask("Can you summarize the video?")
ask("What technologies are discussed?")

Building RAG pipeline for video: Gfr50f6ZBvo
English transcript found!
Total chunks: 207
Vector store created!
Retriever created!
RAG chain created!

✅ RAG Pipeline ready! You can now ask questions.

Question: Can you summarize the video?
Answer: This video features a conversation that touches on various topics related to artificial intelligence, neuroscience, and the evolution of research practices over the past 10 to 20 years. The speakers discuss the challenges of pitching AI concepts to venture capitalists in the past, the importance of reinforcement learning as it relates to how the primate brain learns, and the role of advancements in computing power, particularly through GPUs, in facilitating AI development. They also mention the theoretical frameworks of intelligence and the potential limitations of understanding certain concepts from within the universe. The discussion concludes with reflections on personal habits and changes in research approaches over time.

Question: What t

'The technologies discussed include:\n\n1. **fMRI Machines** - Used for understanding brain architectures and algorithms.\n2. **GPUs** - Highlighted for their importance in compute power, especially in the context of the games industry.\n3. **Deep Learning** - An algorithmic advance credited to researchers like Jeff Hinton.\n4. **Reinforcement Learning** - Emphasized as a scalable approach.\n5. **Transformers** - Mentioned as a significant evolution in AI technology.\n6. **Large Language Models** - Referenced in the context of recent advancements, including GPT-3.\n\nThese technologies contribute to the development and understanding of artificial intelligence.'

In [13]:
!pip install ragas langsmith datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 12.2 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.14.0
    Uninstalling jiter-0.14.0:
      Successfully uninstalled jiter-0.14.0


In [15]:
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "your_langsmith_api_key"  # get from smith.langchain.com
os.environ["LANGCHAIN_PROJECT"] = "YouTube RAG Evaluation"

In [10]:
import os
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

import os
os.environ["OPENAI_API_KEY"] = "your_openai_api_key_here"

# Install correct versions
!pip install -q --upgrade ragas
!pip install -q datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.5/397.5 kB 21.9 MB/s eta 0:00:00


In [11]:
import os
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from datasets import Dataset

# RAGAS latest version imports
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import llm_factory
from openai import OpenAI

# ---- Step 1: Transcript ----
def get_transcript(video_id):
    try:
        ytt = YouTubeTranscriptApi()
        transcript_list = ytt.fetch(video_id)
        transcript = " ".join(chunk.text for chunk in transcript_list)
        print("✅ English transcript found!")
        return transcript
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

video_id = "Gfr50f6ZBvo"
transcript = get_transcript(video_id)
print(f"Transcript length: {len(transcript)} chars")

# ---- Step 2: Split ----
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " "]
)
chunks = splitter.create_documents([transcript])
print(f"✅ Total chunks: {len(chunks)}")

# ---- Step 3: Vector Store ----
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 5})
print("✅ Vector store created!")

# ---- Step 4: RAG Chain ----
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

prompt = PromptTemplate(
    template="""
    You are an expert assistant analyzing a YouTube video transcript.
    Answer ONLY from the provided context.
    If not found, say "This topic was not covered in the video."

    Context: {context}
    Question: {question}
    Answer:
    """,
    input_variables=['context', 'question']
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    RunnableParallel({
        'context': retriever | RunnableLambda(format_docs),
        'question': RunnablePassthrough()
    })
    | prompt | llm | StrOutputParser()
)
print("✅ RAG chain ready!")

# ---- Step 5: Generate answers ----
questions = [
    "What is the main topic of this video?",
    "Who is Demis Hassabis?",
    "What is DeepMind?",
    "What achievements has DeepMind accomplished?",
    "What is the future of AI according to this video?"
]

ground_truths = [
    "The video discusses AI and DeepMind's contributions to science",
    "Demis Hassabis is the CEO and co-founder of DeepMind",
    "DeepMind is an AI research lab owned by Google",
    "DeepMind has achieved breakthroughs in protein folding and game playing",
    "AI will transform science and solve major human problems"
]

answers = []
contexts = []

for q in questions:
    print(f"Processing: {q}")
    retrieved = retriever.invoke(q)
    ctx = [doc.page_content for doc in retrieved]
    ans = chain.invoke(q)
    answers.append(ans)
    contexts.append(ctx)

print("✅ Answers generated!")

# ---- Step 6: RAGAS Evaluation (latest API) ----
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
ragas_llm = llm_factory('gpt-4o-mini', client=openai_client)

# Build evaluation samples
samples = []
for q, a, c, g in zip(questions, answers, contexts, ground_truths):
    samples.append(SingleTurnSample(
        user_input=q,
        response=a,
        retrieved_contexts=c,
        reference=g
    ))

eval_dataset = EvaluationDataset(samples=samples)

# Run evaluation
results = evaluate(
    dataset=eval_dataset,
    metrics=[
        Faithfulness(llm=ragas_llm),
        AnswerRelevancy(llm=ragas_llm),
        ContextPrecision(llm=ragas_llm),
        ContextRecall(llm=ragas_llm)
    ]
)

print("\n--- RAGAS Scores ---")
print(results)
df = results.to_pandas()
print(df)

/tmp/ipykernel_11585/3002509552.py:15: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/tmp/ipykernel_11585/3002509552.py:15: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/tmp/ipykernel_11585/3002509552.py:15: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Faithfulness, 

✅ English transcript found!
Transcript length: 133836 chars
✅ Total chunks: 207
✅ Vector store created!
✅ RAG chain ready!
Processing: What is the main topic of this video?
Processing: Who is Demis Hassabis?
Processing: What is DeepMind?
Processing: What achievements has DeepMind accomplished?
Processing: What is the future of AI according to this video?
✅ Answers generated!


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]


--- RAGAS Scores ---
{'faithfulness': 0.8933, 'answer_relevancy': 0.9638, 'context_precision': 0.6778, 'context_recall': 0.8000}
                                          user_input  \
0              What is the main topic of this video?   
1                             Who is Demis Hassabis?   
2                                  What is DeepMind?   
3       What achievements has DeepMind accomplished?   
4  What is the future of AI according to this video?   

                                  retrieved_contexts  \
0  [ideas in the history of ai but it's also a pl...   
1  [the following is a conversation with demus ha...   
2  [the following is a conversation with demus ha...   
3  [the following is a conversation with demus ha...   
4  [even with with animals high-level animals why...   

                                            response  \
0  The main topic of this video is the exploratio...   
1  Demis Hassabis is the CEO and co-founder of De...   
2  DeepMind is a company tha

In [13]:
!pip install pyngrok nest-asyncio fastapi uvicorn

In [17]:
import os
import uvicorn
import nest_asyncio
from pyngrok import ngrok
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from deep_translator import GoogleTranslator
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from threading import Thread

# ---- API Keys ----
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"
os.environ["NGROK_AUTHTOKEN"] = "3DMgRNBaoziu8t4wz3nQHAt0BmW_3hbqDghbdfDAQMjgXkPoJ"  # get free from ngrok.com

# ---- FastAPI App ----
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

video_store = {}

class VideoRequest(BaseModel):
    video_id: str

class QuestionRequest(BaseModel):
    video_id: str
    question: str

# ---- Get Transcript ----
def get_transcript(video_id):
    try:
        ytt = YouTubeTranscriptApi()
        transcript_list = ytt.fetch(video_id)
        transcript = " ".join(chunk.text for chunk in transcript_list)
        print(f"✅ English transcript found!")
        return transcript
    except Exception:
        try:
            print("English not found. Trying other languages...")
            ytt = YouTubeTranscriptApi()
            available = ytt.list(video_id)
            for t in available:
                print(f"Found: {t.language} ({t.language_code})")
                raw = t.fetch()
                raw_text = " ".join(chunk.text for chunk in raw)
                translator = GoogleTranslator(source='auto', target='en')
                parts = [raw_text[i:i+4500] for i in range(0, len(raw_text), 4500)]
                transcript = " ".join(translator.translate(p) for p in parts)
                print("✅ Translated to English!")
                return transcript
        except Exception as e:
            print(f"❌ Error: {e}")
            return None

# ---- Build RAG Pipeline ----
def build_pipeline(video_id):
    transcript = get_transcript(video_id)
    if not transcript:
        return None

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100,
        separators=["\n\n", "\n", ".", " "]
    )
    chunks = splitter.create_documents([transcript])
    print(f"✅ Total chunks: {len(chunks)}")

    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vector_store = FAISS.from_documents(chunks, embeddings)

    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 5, "fetch_k": 10}
    )

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

    prompt = PromptTemplate(
        template="""
        You are a strict assistant analyzing a YouTube video transcript.
        Answer ONLY using exact information from the context below.
        Do NOT add any outside knowledge.
        If the answer is not in the context, say "This topic was not covered in the video."

        Context: {context}
        Question: {question}
        Answer:
        """,
        input_variables=['context', 'question']
    )

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    chain = (
        RunnableParallel({
            'context': retriever | RunnableLambda(format_docs),
            'question': RunnablePassthrough()
        })
        | prompt | llm | StrOutputParser()
    )

    print(f"✅ RAG pipeline ready!")
    return chain

# ---- API Routes ----
@app.get("/health")
def health():
    return {"status": "running"}

@app.post("/index")
def index_video(req: VideoRequest):
    if req.video_id in video_store:
        return {"status": "already_indexed", "message": "Video already indexed!"}

    print(f"Indexing video: {req.video_id}")
    chain = build_pipeline(req.video_id)

    if not chain:
        return {"status": "error", "message": "Could not get transcript for this video."}

    video_store[req.video_id] = chain
    return {"status": "success", "message": "Video indexed successfully!"}

@app.post("/ask")
def ask_question(req: QuestionRequest):
    if req.video_id not in video_store:
        return {"answer": "⏳ Video not indexed yet. Please wait."}

    chain = video_store[req.video_id]
    answer = chain.invoke(req.question)
    return {"answer": answer}

@app.get("/indexed")
def get_indexed():
    return {"indexed_videos": list(video_store.keys())}

# ---- Start ngrok + uvicorn ----
nest_asyncio.apply()

# Set ngrok auth token
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"\n🚀 Backend is live at: {public_url}")
print(f"📌 Copy this URL and paste it in popup.js as BACKEND_URL\n")

# Run FastAPI
await uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000)).serve()


🚀 Backend is live at: NgrokTunnel: "https://detection-wife-cattail.ngrok-free.dev" -> "http://localhost:8000"
📌 Copy this URL and paste it in popup.js as BACKEND_URL



INFO:     Started server process [11585]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2401:4900:c990:c821:6cb7:431c:b58c:9f94:0 - "GET /health HTTP/1.1" 200 OK
Indexing video: Gfr50f6ZBvo


ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-15' coro=<as_completed.<locals>.sema_coro() done, defined at /usr/local/lib/python3.12/dist-packages/ragas/async_utils.py:75> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_11585/63592198.py", line 139, in <cell line: 0>
    results = evaluate(
              ^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ragas/_analytics.py", line 278, in wrapper
    result = func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ragas/evaluation.py", line 484, in evaluate
    return run(_async_wrapper())
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ragas/async_utils.py", line 156, in run
    return asyncio.run(coro)
          

✅ English transcript found!
✅ Total chunks: 336
✅ RAG pipeline ready!
INFO:     2401:4900:c990:c821:6cb7:431c:b58c:9f94:0 - "GET /health HTTP/1.1" 200 OK
INFO:     2401:4900:c990:c821:6cb7:431c:b58c:9f94:0 - "POST /index HTTP/1.1" 200 OK
INFO:     2401:4900:c990:c821:6cb7:431c:b58c:9f94:0 - "GET /health HTTP/1.1" 200 OK
INFO:     2401:4900:c990:c821:6cb7:431c:b58c:9f94:0 - "POST /index HTTP/1.1" 200 OK
INFO:     2401:4900:c990:c821:6cb7:431c:b58c:9f94:0 - "POST /ask HTTP/1.1" 200 OK
INFO:     2401:4900:c990:c821:6cb7:431c:b58c:9f94:0 - "POST /ask HTTP/1.1" 200 OK
INFO:     2401:4900:c990:c821:6cb7:431c:b58c:9f94:0 - "GET /health HTTP/1.1" 200 OK
Indexing video: pqBIhXOUEY8
✅ English transcript found!
✅ Total chunks: 72
✅ RAG pipeline ready!
INFO:     2401:4900:c990:c821:6cb7:431c:b58c:9f94:0 - "POST /index HTTP/1.1" 200 OK
INFO:     2401:4900:c990:c821:6cb7:431c:b58c:9f94:0 - "POST /ask HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [11585]
